In [1]:
%uv pip install pyarrow
%uv pip install fastparquet

Using Python 3.12.10 environment at: /usr/local
Audited 1 package in 9ms


Note: you may need to restart the kernel to use updated packages.


Using Python 3.12.10 environment at: /usr/local
Audited 1 package in 5ms


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

In [3]:
tagged = pd.read_parquet("/mnt/titles/tagged_track_titles.parquet")
print(len(tagged))
tagged["name_track_estimate"] = tagged["tagged_name_track"].str.replace(r"<[A-Z]+>", "", regex=True)

5026460


In [4]:
import re
import pandas as pd

def extract_tagged_segments(tagged_title):
    if pd.isna(tagged_title):
        return {}

    TAG_PATTERN = r'<(CO|CA|K|I|N|V|D|M|S|O)>([^<]*)'
    extracted = {
        'catalog': [],
        'name': [],
        'composer': [],
        'index': [],
        'key': [],
        'version': [],
        'other': []
    }

    tag_map = {
        'CA': 'catalog',
        'N': 'name',
        'CO': 'composer',
        'I': 'index',
        'K': 'key',
        'V': 'version',
        'O': 'other'
    }

    for match in re.finditer(TAG_PATTERN, tagged_title):
        tag, value = match.group(1), match.group(2).strip()
        if tag in tag_map and value:
            extracted[tag_map[tag]].append(value)

    return extracted


rows = []

for _, row in tagged.iterrows():
    extracted = extract_tagged_segments(row['tagged_name_track'])

    for field in extracted:
        for value in extracted[field]:
            rows.append({
                'track_rowid': row['track_rowid'],
                'type': field,
                'value': value
            })

tagged_fields_table = pd.DataFrame(rows)

tagged_fields_table.to_parquet(
    "/mnt/titles/track_tags.parquet",
    index=False
)

print(f"Saved {len(tagged_fields_table)} rows to /mnt/titles/track_tags.parquet")

Saved 18320983 rows to /mnt/titles/track_tags.parquet
